# MedAgent — QLoRA Fine-tuning (Qwen2.5-7B)
**Optimized for Kaggle T4 GPU**

التعديلات الأساسية مقارنة بالنسخة السابقة:
- `fp16` بدل `bf16` (T4 مفيهاش bfloat16 native)
- `packing=True` لتقليل الـ padding waste
- `num_epochs=1` للـ baseline run (تقدر ترفعه بعدين)
- `LoRA r=16` بدل 32 (أسرع، مناسب لـ first run)
- `eval_steps` و `save_steps` أكبر (أقل توقفات)
- `dataloader_num_workers=2` + `pin_memory=True`
- `group_by_length=True` (padding أقل)

**التوقع:** ~60-90 دقيقة على T4 (بدل 8+ ساعات)


In [ ]:
import os

# Kaggle paths
OUTPUT_DIR = "/kaggle/working/medagent-lora"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"✅ Output: {OUTPUT_DIR}")
print(f"✅ Checkpoints: {CHECKPOINT_DIR}")


In [ ]:
!pip install -q -U transformers accelerate peft trl datasets bitsandbytes sentencepiece
!pip install -q huggingface_hub
print("✅ Packages installed")


In [ ]:
import os
import json
import torch
from pathlib import Path
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"✅ Torch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

# تحقق من نوع الـ GPU عشان نعرف نستخدم fp16 ولا bf16
gpu_name = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ""
USE_BF16 = any(x in gpu_name for x in ["a100", "a10", "h100", "l4", "l40", "rtx 30", "rtx 40", "rtx a"])
USE_FP16 = not USE_BF16
print(f"\n🔧 Precision: {'bf16' if USE_BF16 else 'fp16'} (auto-detected)")


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# للسرعة على T4: استخدم 3B (علق السطر اللي فوق وفعّل ده)
# MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
HF_REPO = "Hossam7asan/medagent-lora"

# LoRA hyperparams (baseline — ابدأ بـ r=16)
LORA_R = 16
LORA_ALPHA = 32              # ضعف الـ r (نسبة 2:1)
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Training hyperparams
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 8    # Effective batch = 16
MAX_SEQ_LENGTH = 768         # medical dialogues need longer context

LEARNING_RATE = 2e-4
NUM_EPOCHS = 1               # ابدأ بـ 1 — لو الـ loss لسه بينزل، زود
WARMUP_STEPS = 10

# Quantization (4-bit NF4) — compute_dtype حسب الـ GPU
compute_dtype = torch.bfloat16 if USE_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("✅ Config ready")
print(f"   LoRA r={LORA_R}, alpha={LORA_ALPHA}, modules={len(LORA_TARGET_MODULES)}")
print(f"   Effective batch = {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Compute dtype = {compute_dtype}")


In [ ]:
# (اختياري) login لـ HuggingFace لو فيه HF_TOKEN في Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Logged in to HuggingFace")
except Exception as e:
    print(f"ℹ️ HF login skipped: {e}")

print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Qwen2.5: استخدم endoftext كـ pad token (مش EOS) عشان نتجنب مشاكل الـ generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

print(f"\n✅ Model loaded")
print(f"   Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
SYSTEM_EN = (
    "You are MedAgent, a bilingual medical triage AI. "
    "Assess symptoms, ask follow-up questions, determine triage level "
    "(emergency/urgent/routine), and recommend next steps. "
    "Always include a 'consult a physician' disclaimer. "
    "Never provide definitive diagnoses or prescribe medications."
)

SYSTEM_AR = (
    "أنت MedAgent، مساعد طبي ثنائي اللغة لتقييم الحالات. "
    "قيّم الأعراض، اسأل أسئلة متابعة، حدد مستوى الإلحاح (طارئ/عاجل/روتيني)، "
    "ثم اقترح الخطوة التالية. أضف دائماً تنبيه استشارة طبيب. "
    "لا تعطِ تشخيصات قاطعة ولا توصف أدوية."
)


def format_chat(patient: str, doctor: str, lang: str = "en") -> dict:
    """Format as Qwen2.5 ChatML."""
    if not patient or not doctor:
        return {"text": ""}

    system = SYSTEM_AR if lang == "ar" else SYSTEM_EN
    text = (
        f"<|im_start|>system\n{system}<|im_end|>\n"
        f"<|im_start|>user\n{patient}<|im_end|>\n"
        f"<|im_start|>assistant\n{doctor}<|im_end|>\n"
    )
    return {"text": text}


def load_training_data(per_source_limit: int = 5000):
    data = []

    # 1. MedQuAD (English)
    try:
        ds = load_dataset("lavita/MedQuAD", split="train", streaming=True)
        count = 0
        for row in ds:
            f = format_chat(row.get("question", ""), row.get("answer", ""))
            if f["text"]:
                data.append(f)
            count += 1
            if count >= per_source_limit:
                break
        print(f"✅ MedQuAD: {len(data)} examples")
    except Exception as e:
        print(f"❌ MedQuAD failed: {e}")

    # 2. PubMedQA (English)
    try:
        start = len(data)
        ds = load_dataset("pubmed_qa", "pqa_labeled", split="train", streaming=True)
        count = 0
        for row in ds:
            q = row.get("question", "")
            a = row.get("long_answer", "") or " ".join(
                row.get("context", {}).get("contexts", [])
            )
            f = format_chat(q, a)
            if f["text"]:
                data.append(f)
            count += 1
            if count >= per_source_limit // 2:
                break
        print(f"✅ PubMedQA: {len(data) - start} examples")
    except Exception as e:
        print(f"❌ PubMedQA failed: {e}")

    # 3. MedAlpaca flashcards (English)
    try:
        start = len(data)
        ds = load_dataset(
            "medalpaca/medical_meadow_medical_flashcards",
            split="train",
            streaming=True,
        )
        count = 0
        for row in ds:
            f = format_chat(row.get("input", ""), row.get("output", ""))
            if f["text"]:
                data.append(f)
            count += 1
            if count >= per_source_limit // 2:
                break
        print(f"✅ MedAlpaca: {len(data) - start} examples")
    except Exception as e:
        print(f"❌ MedAlpaca failed: {e}")

    # 4. (اختياري للـ V2) — Arabic medical data
    # ضيف هنا أي dataset عربي طبي تلاقيه (مثلاً MedDialog-AR لو متاح)
    # أو ترجمة عينة من MedQuAD بـ NLLB-200

    if not data:
        raise RuntimeError("⚠️ All sources failed. Enable Internet in Kaggle settings!")

    import random
    random.seed(42)
    random.shuffle(data)

    n = len(data)
    train = data[:int(n * 0.8)]
    val = data[int(n * 0.8):int(n * 0.9)]
    test = data[int(n * 0.9):]

    print(f"\n📊 Total: {n} | Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
    return train, val, test


print("Loading training data...")
train_data, val_data, test_data = load_training_data(per_source_limit=5000)

# للـ first run: 2000 train + 200 val (سرعة + اختبار)
# لما تتأكد إن كل حاجة شغالة، زود الأرقام
TRAIN_SUBSET = 2000
VAL_SUBSET = 200

train_ds = Dataset.from_list(train_data[:TRAIN_SUBSET])
val_ds = Dataset.from_list(val_data[:VAL_SUBSET])

print(f"\n✅ Using subset — Train: {len(train_ds)} | Val: {len(val_ds)}")


In [ ]:
# Pre-training optimizations
model.config.use_cache = False
# gradient_checkpointing هيتفعل من SFTConfig مش لازم نناديه هنا

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",

    # Logging
    logging_steps=10,
    logging_first_step=True,

    # Saving (أقل تكراراً عشان نوفر وقت I/O)
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    # Evaluation (كل 200 step بدل 100)
    eval_strategy="steps",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # ⚡ 4-bit QLoRA handles precision internally — no mixed precision needed
    # fp16/bf16 in training args conflicts with QLoRA on torch 2.10+
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # ⚡ Data loading optimizations
    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # SFT-specific
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=True,                  # ⚡ بيوفر 30-50% من الوقت

    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("=" * 60)
print("🚀 Starting training...")
print(f"   Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"   Effective batch: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Precision: {compute_dtype} (4-bit QLoRA)")
print(f"   Packing: ON")
print(f"   Expected time: ~60-90 min on T4")
print("=" * 60)

trainer.train()

print("\n✅ Training complete!")


In [ ]:
print("Saving LoRA adapter...")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import subprocess
result = subprocess.run(["du", "-sh", OUTPUT_DIR], capture_output=True, text=True)
print(f"✅ Saved: {OUTPUT_DIR}")
print(f"   Size: {result.stdout.strip()}")

print("\nFiles:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f"   {f} ({size:.1f} MB)")


In [ ]:
# لو login تم في cell 4 خلاص، ده هيشتغل مباشرة
try:
    print(f"Pushing to {HF_REPO}...")
    model.push_to_hub(HF_REPO, private=True)
    tokenizer.push_to_hub(HF_REPO, private=True)
    print(f"✅ Pushed to: https://huggingface.co/{HF_REPO}")
except Exception as e:
    print(f"⚠️ Push failed: {e}")
    print("   تأكد إن HF_TOKEN موجود في Kaggle Secrets وعنده write permission")


In [ ]:
def test_model():
    """Quick test of the fine-tuned model (English + Arabic)."""
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=200,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
    )

    test_prompts = [
        # EN — Emergency
        f"<|im_start|>system\n{SYSTEM_EN}<|im_end|>\n"
        "<|im_start|>user\nI have a severe headache and blurred vision, "
        "started suddenly.<|im_end|>\n<|im_start|>assistant\n",

        # AR — Emergency
        f"<|im_start|>system\n{SYSTEM_AR}<|im_end|>\n"
        "<|im_start|>user\nعندي ألم شديد في الصدر وصعوبة في التنفس<|im_end|>\n"
        "<|im_start|>assistant\n",

        # EN — Routine
        f"<|im_start|>system\n{SYSTEM_EN}<|im_end|>\n"
        "<|im_start|>user\nMild cough for 3 days, no fever.<|im_end|>\n"
        "<|im_start|>assistant\n",
    ]

    for i, prompt in enumerate(test_prompts, 1):
        result = pipe(prompt)[0]["generated_text"]
        response = result[len(prompt):]
        user_msg = prompt.split("<|im_start|>user\n")[1].split("<|im_end|>")[0]
        print(f"\n{'=' * 60}")
        print(f"Test {i}")
        print(f"{'=' * 60}")
        print(f"📥 Input:  {user_msg[:150]}")
        print(f"📤 Output: {response[:400]}")


test_model()
print("\n🎉 All done!")


## 📝 ملاحظات للخطوات الجاية

### بعد ما تتأكد إن أول run شغال:

**1. زود البيانات:**
```python
TRAIN_SUBSET = 10000   # بدل 2000
VAL_SUBSET = 1000
```

**2. زود الـ epochs لو الـ loss لسه بينزل:**
```python
NUM_EPOCHS = 2  # أو 3
```

**3. لو عايز quality أعلى، زود LoRA rank:**
```python
LORA_R = 32
LORA_ALPHA = 64
```

### للـ V2 (إضافة العربي):
- استخدم **NLLB-200-distilled-600M** لترجمة عينة من MedQuAD لعربي
- أو ابحث عن **MedDialog-AR** على HuggingFace
- أو عينة محدودة من **Tarjama-25** (حوالي 25K cross-lingual sample)

### لحفظ الـ adapter كـ Kaggle Dataset:
بعد ما الـ session يخلص: `Output panel → Save as Dataset → اسمه medagent-lora-v1`

### ربط بـ MedAgent backend:
```python
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", ...)
model = PeftModel.from_pretrained(base, "Hossam7asan/medagent-lora")
```
